### Least Squares via CG without forming $Z^T Z$

We want to fit coefficients $a\in\mathbb{R}^m$ to data $y\in\mathbb{R}^n$ using an overdetermined linear model

$$
y \approx Za,\qquad Z\in\mathbb{R}^{n\times m},\; n>m.
$$

The least-squares solution minimizes

$$
\phi(a)=\|Za-y\|_2^2.
$$

The optimality condition is obtained by setting the gradient to zero:

$$
\nabla \phi(a)=2Z^T( Za-y)=0
\quad\Rightarrow\quad
(Z^T Z)a = Z^T y.
$$

Rather than forming the matrix $Z^T Z$ explicitly (which can be expensive and can worsen conditioning),
we apply **Conjugate Gradient (CG)** using only the operations:

- given $v$, compute $Z v$
- then compute $Z^T(Zv)$

This avoids building $Z^T Z$ and only requires matrix-vector products.

In [2]:
import numpy as np

def cgnr(Z, y, maxiter=200, tol=1e-10, verbose=True):
    """
    CGNR (Conjugate Gradient on the Normal Residual):
    Solves min ||Z a - y||_2 using CG on (Z^T Z) a = Z^T y
    WITHOUT forming Z^T Z explicitly.

    Inputs:
      Z: (n,m) matrix or LinearOperator-like with matvec and rmatvec
      y: (n,) data
    Returns:
      a: (m,) least-squares solution estimate
      history: list of residual norms ||Z a - y||
    """

    # Allow either a NumPy matrix or an operator with .matvec/.rmatvec
    if isinstance(Z, np.ndarray):
        matvec  = lambda v: Z @ v
        rmatvec = lambda w: Z.T @ w
        m = Z.shape[1]
    else:
        matvec  = Z.matvec
        rmatvec = Z.rmatvec
        m = Z.shape[1]

    a = np.zeros(m)                 # initial guess
    r = y - matvec(a)               # residual in data space (n,)
    g = rmatvec(r)                  # gradient-like vector in model space (m,)
    p = g.copy()

    g_norm2 = np.dot(g, g)
    history = [np.linalg.norm(r)]

    if verbose:
        print(f"iter 0: ||r|| = {history[-1]:.3e}")

    for k in range(1, maxiter + 1):
        # q = (Z^T Z) p = Z^T( Z p ) computed as two matvecs
        Zp = matvec(p)              # (n,)
        q  = rmatvec(Zp)            # (m,)

        denom = np.dot(p, q)
        if denom == 0:
            break

        alpha = g_norm2 / denom

        a = a + alpha * p
        r = r - alpha * Zp          # update residual in data space
        g_new = rmatvec(r)          # new gradient-like vector

        g_new_norm2 = np.dot(g_new, g_new)
        beta = g_new_norm2 / g_norm2

        p = g_new + beta * p
        g = g_new
        g_norm2 = g_new_norm2

        history.append(np.linalg.norm(r))
        if verbose and (k <= 10 or k % 10 == 0):
            print(f"iter {k:3d}: ||r|| = {history[-1]:.3e}")

        if history[-1] <= tol * history[0]:
            if verbose:
                print(f"Converged at iter {k} (relative residual <= {tol}).")
            break

    return a, history


In [3]:


# -------------------------
# Demo with synthetic data
# -------------------------
np.random.seed(0)

n = 300     # data size
m = 40      # number of coefficients (n > m)

Z = np.random.randn(n, m)
a_true = np.random.randn(m)

noise_level = 0.05
y = Z @ a_true + noise_level * np.random.randn(n)

a_est, hist = cgnr(Z, y, maxiter=200, tol=1e-12, verbose=True)

print("\nRelative coefficient error ||a_est - a_true|| / ||a_true|| =",
      np.linalg.norm(a_est - a_true) / np.linalg.norm(a_true))

# Compare with NumPy's least squares (for validation only)
a_np, *_ = np.linalg.lstsq(Z, y, rcond=None)
print("Relative error vs np.linalg.lstsq solution =",
      np.linalg.norm(a_est - a_np) / np.linalg.norm(a_np))

iter 0: ||r|| = 1.133e+02
iter   1: ||r|| = 3.446e+01
iter   2: ||r|| = 1.061e+01
iter   3: ||r|| = 3.546e+00
iter   4: ||r|| = 1.486e+00
iter   5: ||r|| = 9.595e-01
iter   6: ||r|| = 8.872e-01
iter   7: ||r|| = 8.775e-01
iter   8: ||r|| = 8.766e-01
iter   9: ||r|| = 8.765e-01
iter  10: ||r|| = 8.765e-01
iter  20: ||r|| = 8.765e-01
iter  30: ||r|| = 8.765e-01
iter  40: ||r|| = 8.765e-01
iter  50: ||r|| = 8.765e-01
iter  60: ||r|| = 8.765e-01
iter  70: ||r|| = 8.765e-01
iter  80: ||r|| = 8.765e-01
iter  90: ||r|| = 8.765e-01
iter 100: ||r|| = 8.765e-01
iter 110: ||r|| = 8.765e-01
iter 120: ||r|| = 8.765e-01
iter 130: ||r|| = 8.765e-01
iter 140: ||r|| = 8.765e-01
iter 150: ||r|| = 8.768e-01
iter 160: ||r|| = 9.768e-01
iter 170: ||r|| = 7.434e+00
iter 180: ||r|| = 1.306e+02
iter 190: ||r|| = 2.162e+03
iter 200: ||r|| = 3.911e+04

Relative coefficient error ||a_est - a_true|| / ||a_true|| = 416.28836132897686
Relative error vs np.linalg.lstsq solution = 416.2161684589489


CGNR/LSCG never builds $Z^T Z$; 

it only needs “apply $Z$” and “apply $Z^T$”.

This is exactly what you want when $Z$ is huge or when $Z$ is an implicit operator (FFT, convolution, PDE modeling, etc.).